Email Dataset Generation Notebook

Instructions

1. **Run cells sequentially** - Start from the top and work your way down
2. **Check output at each step** - Review generated emails before proceeding to next combination
3. **Model**: Using `qwen2.5:14b-instruct-q4_K_M` via Ollama
4. **Total**: 21 combinations × 5 emails each = 105 emails

Structure

- **Cells 1-5**: Setup and helper functions
- **Cell 6+**: Individual combination cells (run one at a time, check output)
- **Last cell**: Save all results to JSONL file

Requirements

- Ollama running: `ollama serve`
- Model available: `qwen2.5:14b-instruct-q4_K_M`
- Each email: 400-500 characters
- All replies end with: "Best Regards,\nZubair"


In [1]:
# Setup: Import libraries and define personal information

import json
import uuid
import ollama
from typing import Dict, List
import time

# Personal Information
PERSONAL_INFO = {
    "ds_resume": "https://drive.google.com/file/d/16lp87m0BbAx4uTPcdHGCIDHEdSQtywy0/view?usp=sharing",
    "swe_resume": "https://drive.google.com/file/d/1NsbClvTx1MgSHhjvdcdpdTppI5MXxtYQ/view?usp=sharing",
    "linkedin": "https://www.linkedin.com/in/zubair-atha/",
    "portfolio": "https://zubairatha.vercel.app/",
    "github": "https://github.com/zubairatha",
    "phone": "+16463925601",
    "zoom_link": "https://us05web.zoom.us/j/5756631049?pwd=2whWm7gb5MHL5GIspFbviES0GQPuyE.1",
    "zoom_meeting_id": "575 663 1049",
    "zoom_passcode": "4Jp5UP",
    "calendly": "https://calendly.com/za2366-columbia/30min"
}

MODEL_NAME = "qwen2.5:14b-instruct-q4_K_M"

print("✅ Setup complete")
print(f"Model: {MODEL_NAME}")
print(f"Personal info loaded: {len(PERSONAL_INFO)} items")


✅ Setup complete
Model: qwen2.5:14b-instruct-q4_K_M
Personal info loaded: 10 items


In [2]:
# Initialize storage for results
all_results = {}
all_emails_formatted = []  # Will store final formatted emails with IDs and labels

def generate_emails(prompt: str, combination_num: int) -> Dict:
    """
    Generate 5 email pairs using Ollama.
    Returns dict with email_1 through email_5.
    """
    print(f"\n🔄 Generating emails for Combination {combination_num}...")
    
    # Enhance prompt with length requirement
    enhanced_prompt = prompt + "\n\nCRITICAL: Both prospect_email and reply MUST be between 400-500 characters each. Count characters carefully and ensure they meet this requirement."
    
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert at generating realistic email conversations. Always output valid JSON format. CRITICAL: prospect_email and reply fields must each be 400-500 characters long. Count characters and ensure they meet this exact requirement."
                },
                {
                    "role": "user",
                    "content": enhanced_prompt
                }
            ],
            options={
                "temperature": 0.7,
                "top_p": 0.9
            }
        )
        
        content = response["message"]["content"]
        
        # Try to extract JSON from response
        # Handle code blocks if present
        if "```json" in content:
            json_start = content.find("```json") + 7
            json_end = content.find("```", json_start)
            content = content[json_start:json_end].strip()
        elif "```" in content:
            json_start = content.find("```") + 3
            json_end = content.find("```", json_start)
            content = content[json_start:json_end].strip()
        
        # Parse JSON
        result = json.loads(content)
        
        print(f"✅ Generated {len(result)} emails")
        return result
        
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
        print(f"Response content:\n{content[:500]}")
        return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None


def validate_email_pair(email_data: Dict, expected_intents: List[str], expected_artifacts: List[str]) -> bool:
    """
    Validate that email pair meets requirements.
    """
    required_keys = ["subject", "sender_email", "prospect_email", "reply"]
    
    # Check all keys exist
    for key in required_keys:
        if key not in email_data:
            print(f"  ❌ Missing key: {key}")
            return False
    
    # Check character limits (400-500)
    prospect_len = len(email_data["prospect_email"])
    reply_len = len(email_data["reply"])
    
    if prospect_len < 400 or prospect_len > 500:
        print(f"  ⚠️  Prospect email length: {prospect_len} (should be 400-500)")
    
    if reply_len < 400 or reply_len > 500:
        print(f"  ⚠️  Reply length: {reply_len} (should be 400-500)")
    
    # Check reply ends correctly
    if not email_data["reply"].strip().endswith("Zubair"):
        print(f"  ⚠️  Reply doesn't end with 'Zubair'")
    
    # Check artifacts are mentioned in reply (skip if artifacts list is empty/None)
    if expected_artifacts and len(expected_artifacts) > 0:
        reply_lower = email_data["reply"].lower()
        for artifact in expected_artifacts:
            if artifact == "calendly":
                if "calendly.com" not in reply_lower:
                    print(f"  ⚠️  Calendly link not found in reply")
            elif artifact == "ds_resume":
                if "drive.google.com" not in reply_lower or "16lp87m0BbAx4uTPcdHGCIDHEdSQtywy0" not in reply_lower:
                    print(f"  ⚠️  DS resume link not found in reply")
            elif artifact == "swe_resume":
                if "drive.google.com" not in reply_lower or "1NsbClvTx1MgSHhjvdcdpdTppI5MXxtYQ" not in reply_lower:
                    print(f"  ⚠️  SWE resume link not found in reply")
            elif artifact == "linkedin_profile":
                if "linkedin.com/in/zubair-atha" not in reply_lower:
                    print(f"  ⚠️  LinkedIn link not found in reply")
            elif artifact == "portfolio":
                if "zubairatha.vercel.app" not in reply_lower:
                    print(f"  ⚠️  Portfolio link not found in reply")
            elif artifact == "github":
                if "github.com/zubairatha" not in reply_lower:
                    print(f"  ⚠️  GitHub link not found in reply")
            elif artifact == "zoom_link":
                if "zoom.us/j/5756631049" not in reply_lower:
                    print(f"  ⚠️  Zoom link not found in reply")
            elif artifact == "phone_number":
                if "+16463925601" not in reply_lower:
                    print(f"  ⚠️  Phone number not found in reply")
    
    return True

def format_email_for_output(email_data: Dict, topic: str, intents: List[str], artifacts: List[str]) -> Dict:
    """
    Format email pair with ID and labels for final output.
    """
    return {
        "id": str(uuid.uuid4()),
        "labels": {
            "topic": topic,
            "intents": intents,
            "artifacts": artifacts
        },
        "subject": email_data["subject"],
        "sender_email": email_data["sender_email"],
        "prospect_email": email_data["prospect_email"],
        "reply": email_data["reply"]
    }

def process_combination(prompt: str, combination_num: int, topic: str, intents: List[str], artifacts: List[str]):
    """
    Generate, validate, and format emails for a combination.
    Returns list of formatted emails for this combination.
    """
    result = generate_emails(prompt, combination_num)
    combination_formatted_emails = []  # Store formatted emails for this combination only
    
    if result:
        print(f"\n📋 Validating Combination {combination_num}...")
        valid_count = 0
        for i in range(1, 6):
            email_key = f"email_{i}"
            if email_key in result:
                print(f"\nEmail {i}:")
                if validate_email_pair(result[email_key], intents, artifacts):
                    valid_count += 1
                    formatted = format_email_for_output(
                        result[email_key],
                        topic=topic,
                        intents=intents,
                        artifacts=artifacts
                    )
                    combination_formatted_emails.append(formatted)
                    all_emails_formatted.append(formatted)
            else:
                print(f"❌ Missing {email_key}")
        
        print(f"\n✅ Valid emails: {valid_count}/5")
        
        all_results[f"combination_{combination_num}"] = result
        print(f"\n💾 Total emails collected: {len(all_emails_formatted)}")
        
        # Return formatted emails for this combination
        return combination_formatted_emails
    else:
        print(f"❌ Generation failed for Combination {combination_num}")
        return []

print("✅ All helper functions defined")


✅ All helper functions defined


In [3]:
# Combination 1: Professor/Academic | schedule | calendly

COMBINATION_1_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Professor/Academic
INTENT: schedule (professor wants to schedule a meeting with student Zubair)
ARTIFACT: calendly (Zubair shares his Calendly link)

INCOMING EMAIL GUIDELINES:
- Professor/academic reaching out TO Zubair (a student) to schedule a meeting
- Address Zubair as "Zubair" or "Dear Zubair" (NOT "Professor Zubair" or "Dr. Zubair")
- Could be about research collaboration, project discussion, thesis guidance, etc.
- Professional but friendly tone appropriate for professor-to-student communication
- 400-500 characters
- Include sender name and email (university domain)

REPLY GUIDELINES:
- Zubair (the student) responds professionally and respectfully
- Shares Calendly link: https://calendly.com/za2366-columbia/30min
- Mentions availability or flexibility
- Tone should be appropriate for a student responding to a professor
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {
    "subject": "...",
    "sender_email": "...",
    "prospect_email": "...",
    "reply": "..."
  },
  "email_2": {
    "subject": "...",
    "sender_email": "...",
    "prospect_email": "...",
    "reply": "..."
  },
  "email_3": {
    "subject": "...",
    "sender_email": "...",
    "prospect_email": "...",
    "reply": "..."
  },
  "email_4": {
    "subject": "...",
    "sender_email": "...",
    "prospect_email": "...",
    "reply": "..."
  },
  "email_5": {
    "subject": "...",
    "sender_email": "...",
    "prospect_email": "...",
    "reply": "..."
  }
}
"""

# Process combination using helper function
combination_1_results = process_combination(
    COMBINATION_1_PROMPT,
    combination_num=1,
    topic="Professor/Academic",
    intents=["schedule"],
    artifacts=["calendly"]
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 1 - ALL FORMATTED EMAILS ({len(combination_1_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_1_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 1...
✅ Generated 5 emails

📋 Validating Combination 1...

Email 1:
  ⚠️  Prospect email length: 161 (should be 400-500)
  ⚠️  Reply length: 293 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 128 (should be 400-500)
  ⚠️  Reply length: 231 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 124 (should be 400-500)
  ⚠️  Reply length: 208 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 110 (should be 400-500)
  ⚠️  Reply length: 195 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 130 (should be 400-500)
  ⚠️  Reply length: 195 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 5

📧 COMBINATION 1 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "eef2f02b-5d07-45dd-9098-b4fe094cfd29",
  "labels": {
    "topic": "Professor/Academic",
    "intents": [
      "schedule"
    ],
    "artifacts": [
      "calendly"
    ]
  },
  "subject": "Meeting to Discuss Research Collaboration",
  "sende

In [4]:
# Results storage is initialized in Cell 2
# This cell is kept for reference - all_results and all_emails_formatted are already defined
print("✅ Results storage ready (initialized in Cell 2)")


✅ Results storage ready (initialized in Cell 2)


In [5]:
# format_email_for_output is defined in Cell 2
# This cell is kept for reference
print("✅ All functions are defined in Cell 2")


✅ All functions are defined in Cell 2


In [6]:
# process_combination is defined in Cell 2
# This cell is kept for reference
print("✅ All functions are defined in Cell 2")


✅ All functions are defined in Cell 2


In [7]:
# Combination 2: Professor/Academic | request_feedback | draft

COMBINATION_2_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Professor/Academic
INTENT: request_feedback (professor requests feedback from student Zubair on something)
ARTIFACT: draft (Zubair sends a draft document)

INCOMING EMAIL GUIDELINES:
- Professor asking Zubair (the student) for feedback on a draft, proposal, or document
- Address Zubair as "Zubair" or "Dear Zubair" (NOT "Professor Zubair" or "Dr. Zubair")
- Could be research proposal, paper draft, project outline, etc.
- Professional academic tone appropriate for professor-to-student communication
- 400-500 characters
- Include sender name and email (university domain)

REPLY GUIDELINES:
- Zubair (the student) acknowledges the request respectfully
- Mentions sending a draft document (be specific: research draft, proposal draft, etc.)
- Offers to discuss further
- Tone should be appropriate for a student responding to a professor
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

# Process combination
combination_2_results = process_combination(
    COMBINATION_2_PROMPT,
    combination_num=2,
    topic="Professor/Academic",
    intents=["request_feedback"],
    artifacts=["draft"]
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 2 - ALL FORMATTED EMAILS ({len(combination_2_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_2_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 2...
✅ Generated 5 emails

📋 Validating Combination 2...

Email 1:
  ⚠️  Prospect email length: 203 (should be 400-500)
  ⚠️  Reply length: 265 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 174 (should be 400-500)
  ⚠️  Reply length: 247 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 165 (should be 400-500)
  ⚠️  Reply length: 232 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 158 (should be 400-500)
  ⚠️  Reply length: 239 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 168 (should be 400-500)
  ⚠️  Reply length: 233 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 10

📧 COMBINATION 2 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "b4710263-83c4-495a-993d-554e816c0980",
  "labels": {
    "topic": "Professor/Academic",
    "intents": [
      "request_feedback"
    ],
    "artifacts": [
      "draft"
    ]
  },
  "subject": "Feedback Requested on Research Proposal",
  "s

In [8]:
# Combination 3: Professor/Academic | accept_or_decline | ds_resume

COMBINATION_3_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Professor/Academic
INTENT: accept_or_decline (professor offers position/opportunity to student Zubair, Zubair accepts or declines)
ARTIFACT: ds_resume (Zubair shares his Data Science resume)

INCOMING EMAIL GUIDELINES:
- Professor offering a research position, RA position, or academic opportunity TO Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair" (NOT "Professor Zubair" or "Dr. Zubair")
- Mentions reviewing application/resume
- Professional offer tone appropriate for professor-to-student communication
- 400-500 characters
- Include sender name and email (university domain)

REPLY GUIDELINES:
- Zubair (the student) accepts or declines (mix both - some accept, some decline)
- If accepting: expresses excitement, shares DS resume link
- If declining: polite decline, may still share resume for future
- DS Resume link: https://drive.google.com/file/d/16lp87m0BbAx4uTPcdHGCIDHEdSQtywy0/view?usp=sharing
- Tone should be appropriate for a student responding to a professor
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_3_results = process_combination(
    COMBINATION_3_PROMPT,
    combination_num=3,
    topic="Professor/Academic",
    intents=["accept_or_decline"],
    artifacts=["ds_resume"]
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 3 - ALL FORMATTED EMAILS ({len(combination_3_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_3_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 3...
✅ Generated 5 emails

📋 Validating Combination 3...

Email 1:
  ⚠️  Prospect email length: 312 (should be 400-500)
  ⚠️  Reply length: 343 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 2:
  ⚠️  Prospect email length: 236 (should be 400-500)
  ⚠️  Reply length: 251 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 3:
  ⚠️  Prospect email length: 220 (should be 400-500)
  ⚠️  Reply length: 356 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 4:
  ⚠️  Prospect email length: 201 (should be 400-500)
  ⚠️  Reply length: 364 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 5:
  ⚠️  Prospect email length: 230 (should be 400-500)
  ⚠️  DS resume link not found in reply

✅ Valid emails: 5/5

💾 Total emails collected: 15

📧 COMBINATION 3 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "d31dedad-5444-4977-bc3d-bd713ff990ac",
  "labels": {
    "topic": "Professor/Academic

In [9]:
# Combination 4: Professor/Academic | reschedule | zoom_link

COMBINATION_4_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Professor/Academic
INTENT: reschedule (professor wants to reschedule a meeting with student Zubair)
ARTIFACT: zoom_link (Zubair provides Zoom link for rescheduled meeting)

INCOMING EMAIL GUIDELINES:
- Professor requesting to reschedule an existing meeting with Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair" (NOT "Professor Zubair" or "Dr. Zubair")
- Mentions reason (conflict, emergency, etc.)
- Professional tone appropriate for professor-to-student communication
- 400-500 characters
- Include sender name and email (university domain)

REPLY GUIDELINES:
- Zubair (the student) acknowledges reschedule request respectfully
- Confirms new time or suggests alternatives
- Shares Zoom link: https://us05web.zoom.us/j/5756631049?pwd=2whWm7gb5MHL5GIspFbviES0GQPuyE.1
- Mentions Meeting ID: 575 663 1049 and Passcode: 4Jp5UP
- Tone should be appropriate for a student responding to a professor
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_4_results = process_combination(
    COMBINATION_4_PROMPT,
    combination_num=4,
    topic="Professor/Academic",
    intents=["reschedule"],
    artifacts=["zoom_link"]
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 4 - ALL FORMATTED EMAILS ({len(combination_4_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_4_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 4...
✅ Generated 5 emails

📋 Validating Combination 4...

Email 1:
  ⚠️  Prospect email length: 180 (should be 400-500)
  ⚠️  Reply length: 305 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 228 (should be 400-500)
  ⚠️  Reply length: 303 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 225 (should be 400-500)
  ⚠️  Reply length: 333 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 198 (should be 400-500)
  ⚠️  Reply length: 259 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 219 (should be 400-500)
  ⚠️  Reply length: 288 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 20

📧 COMBINATION 4 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "4936b047-db5d-42ac-bc17-eab9f8ba0c72",
  "labels": {
    "topic": "Professor/Academic",
    "intents": [
      "reschedule"
    ],
    "artifacts": [
      "zoom_link"
    ]
  },
  "subject": "Request to Reschedule Our Meeting",
  "sender_em

In [10]:
# Combination 5: Professor/Academic | send_materials, request_feedback | report, draft

COMBINATION_5_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Professor/Academic
INTENTS: send_materials, request_feedback (professor requests materials AND feedback from student Zubair)
ARTIFACTS: report, draft (Zubair sends both a report and draft)

INCOMING EMAIL GUIDELINES:
- Professor requesting both materials (report) and feedback on a draft FROM Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair" (NOT "Professor Zubair" or "Dr. Zubair")
- Could be research progress report + draft paper, project report + proposal draft, etc.
- Professional academic tone appropriate for professor-to-student communication
- 400-500 characters
- Include sender name and email (university domain)

REPLY GUIDELINES:
- Zubair (the student) acknowledges both requests respectfully
- Mentions sending report and draft document
- Offers to discuss further
- Tone should be appropriate for a student responding to a professor
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_5_results = process_combination(
    COMBINATION_5_PROMPT,
    combination_num=5,
    topic="Professor/Academic",
    intents=['send_materials', 'request_feedback'],
    artifacts=['report', 'draft']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 5 - ALL FORMATTED EMAILS ({len(combination_5_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_5_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 5...
✅ Generated 5 emails

📋 Validating Combination 5...

Email 1:
  ⚠️  Prospect email length: 247 (should be 400-500)
  ⚠️  Reply length: 354 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 232 (should be 400-500)
  ⚠️  Reply length: 338 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 214 (should be 400-500)
  ⚠️  Reply length: 390 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 213 (should be 400-500)
  ⚠️  Reply length: 349 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 206 (should be 400-500)
  ⚠️  Reply length: 333 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 25

📧 COMBINATION 5 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "ae9dd512-7f32-4d71-99f7-c207d0b1a5e5",
  "labels": {
    "topic": "Professor/Academic",
    "intents": [
      "send_materials",
      "request_feedback"
    ],
    "artifacts": [
      "report",
      "draft"
    ]
  },
  "subject": "Reques

In [11]:
# Combination 6: Recruiter/Job Search | send_materials | swe_resume

COMBINATION_6_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Recruiter/Job Search
INTENT: send_materials (recruiter requests resume/materials from student Zubair)
ARTIFACT: swe_resume (Zubair shares Software Engineering resume)

INCOMING EMAIL GUIDELINES:
- Recruiter from tech company reaching out TO Zubair (the student) about software engineering position
- Address Zubair as "Zubair" or "Dear Zubair"
- Requests resume or application materials
- Professional recruiter tone
- 400-500 characters
- Include sender name, company, and email (company domain)

REPLY GUIDELINES:
- Zubair (the student) responds professionally
- Shares SWE resume link: https://drive.google.com/file/d/1NsbClvTx1MgSHhjvdcdpdTppI5MXxtYQ/view?usp=sharing
- Expresses interest in the opportunity
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_6_results = process_combination(
    COMBINATION_6_PROMPT,
    combination_num=6,
    topic="Recruiter/Job Search",
    intents=['send_materials'],
    artifacts=['swe_resume']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 6 - ALL FORMATTED EMAILS ({len(combination_6_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_6_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 6...
✅ Generated 5 emails

📋 Validating Combination 6...

Email 1:
  ⚠️  Prospect email length: 189 (should be 400-500)
  ⚠️  Reply length: 289 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 2:
  ⚠️  Prospect email length: 191 (should be 400-500)
  ⚠️  Reply length: 288 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 3:
  ⚠️  Prospect email length: 200 (should be 400-500)
  ⚠️  Reply length: 299 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 4:
  ⚠️  Prospect email length: 169 (should be 400-500)
  ⚠️  Reply length: 317 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 5:
  ⚠️  Prospect email length: 173 (should be 400-500)
  ⚠️  Reply length: 312 (should be 400-500)
  ⚠️  SWE resume link not found in reply

✅ Valid emails: 5/5

💾 Total emails collected: 30

📧 COMBINATION 6 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "84834268-230e-40b8-abd0-5b2c05dee650

In [12]:
# Combination 7: Recruiter/Job Search | send_materials | ds_resume

COMBINATION_7_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Recruiter/Job Search
INTENT: send_materials (recruiter requests resume/materials from student Zubair)
ARTIFACT: ds_resume (Zubair shares Data Science resume)

INCOMING EMAIL GUIDELINES:
- Recruiter from company reaching out TO Zubair (the student) about data science/ML position
- Address Zubair as "Zubair" or "Dear Zubair"
- Requests resume or application materials
- Professional recruiter tone
- 400-500 characters
- Include sender name, company, and email (company domain)

REPLY GUIDELINES:
- Zubair (the student) responds professionally
- Shares DS resume link: https://drive.google.com/file/d/16lp87m0BbAx4uTPcdHGCIDHEdSQtywy0/view?usp=sharing
- Expresses interest in the data science opportunity
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_7_results = process_combination(
    COMBINATION_7_PROMPT,
    combination_num=7,
    topic="Recruiter/Job Search",
    intents=['send_materials'],
    artifacts=['ds_resume']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 7 - ALL FORMATTED EMAILS ({len(combination_7_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_7_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 7...
✅ Generated 5 emails

📋 Validating Combination 7...

Email 1:
  ⚠️  Prospect email length: 226 (should be 400-500)
  ⚠️  Reply length: 312 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 2:
  ⚠️  Prospect email length: 214 (should be 400-500)
  ⚠️  Reply length: 300 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 3:
  ⚠️  Prospect email length: 222 (should be 400-500)
  ⚠️  Reply length: 291 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 4:
  ⚠️  Prospect email length: 235 (should be 400-500)
  ⚠️  Reply length: 303 (should be 400-500)
  ⚠️  DS resume link not found in reply

Email 5:
  ⚠️  Prospect email length: 193 (should be 400-500)
  ⚠️  Reply length: 289 (should be 400-500)
  ⚠️  DS resume link not found in reply

✅ Valid emails: 5/5

💾 Total emails collected: 35

📧 COMBINATION 7 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "4cf5eb70-1416-4980-b59f-df6152af5964",
  

In [13]:
# Combination 8: Recruiter/Job Search | request_info | linkedin_profile

COMBINATION_8_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Recruiter/Job Search
INTENT: request_info (recruiter wants more information about student Zubair)
ARTIFACT: linkedin_profile (Zubair shares LinkedIn profile)

INCOMING EMAIL GUIDELINES:
- Recruiter interested in learning more about Zubair's (the student's) background
- Address Zubair as "Zubair" or "Dear Zubair"
- Asks for LinkedIn profile or professional information
- Professional recruiter tone
- 400-500 characters
- Include sender name, company, and email (company domain)

REPLY GUIDELINES:
- Zubair (the student) responds professionally
- Shares LinkedIn profile: https://www.linkedin.com/in/zubair-atha/
- Mentions willingness to discuss further
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_8_results = process_combination(
    COMBINATION_8_PROMPT,
    combination_num=8,
    topic="Recruiter/Job Search",
    intents=['request_info'],
    artifacts=['linkedin_profile']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 8 - ALL FORMATTED EMAILS ({len(combination_8_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_8_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 8...
✅ Generated 5 emails

📋 Validating Combination 8...

Email 1:
  ⚠️  Prospect email length: 297 (should be 400-500)
  ⚠️  Reply length: 263 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 293 (should be 400-500)
  ⚠️  Reply length: 318 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 271 (should be 400-500)
  ⚠️  Reply length: 312 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 279 (should be 400-500)
  ⚠️  Reply length: 332 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 246 (should be 400-500)
  ⚠️  Reply length: 312 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 40

📧 COMBINATION 8 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "06a290f4-8937-43a3-9b8c-41d55d212e51",
  "labels": {
    "topic": "Recruiter/Job Search",
    "intents": [
      "request_info"
    ],
    "artifacts": [
      "linkedin_profile"
    ]
  },
  "subject": "Opportunity at TechCorp - We'd Love t

In [14]:
# Combination 9: Recruiter/Job Search | schedule | zoom_link

COMBINATION_9_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Recruiter/Job Search
INTENT: schedule (recruiter wants to schedule interview/call with student Zubair)
ARTIFACT: zoom_link (Zubair provides Zoom link)

INCOMING EMAIL GUIDELINES:
- Recruiter wants to schedule an interview or phone call with Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair"
- Mentions next steps in hiring process
- Professional recruiter tone
- 400-500 characters
- Include sender name, company, and email (company domain)

REPLY GUIDELINES:
- Zubair (the student) confirms availability
- Shares Zoom link: https://us05web.zoom.us/j/5756631049?pwd=2whWm7gb5MHL5GIspFbviES0GQPuyE.1
- Mentions Meeting ID: 575 663 1049 and Passcode: 4Jp5UP
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_9_results = process_combination(
    COMBINATION_9_PROMPT,
    combination_num=9,
    topic="Recruiter/Job Search",
    intents=['schedule'],
    artifacts=['zoom_link']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 9 - ALL FORMATTED EMAILS ({len(combination_9_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_9_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 9...
✅ Generated 5 emails

📋 Validating Combination 9...

Email 1:
  ⚠️  Prospect email length: 214 (should be 400-500)
  ⚠️  Reply length: 248 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 214 (should be 400-500)
  ⚠️  Reply length: 288 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 228 (should be 400-500)
  ⚠️  Reply length: 273 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 211 (should be 400-500)
  ⚠️  Reply length: 273 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 212 (should be 400-500)
  ⚠️  Reply length: 281 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 45

📧 COMBINATION 9 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "3244a012-9d6e-4ea5-b6e3-e3fceb512dd0",
  "labels": {
    "topic": "Recruiter/Job Search",
    "intents": [
      "schedule"
    ],
    "artifacts": [
      "zoom_link"
    ]
  },
  "subject": "Next Steps for Interview with ACME Inc.",
  "sen

In [15]:
# Combination 10: Recruiter/Job Search | send_materials | swe_resume, linkedin_profile

COMBINATION_10_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Recruiter/Job Search
INTENT: send_materials (recruiter requests materials from student Zubair)
ARTIFACTS: swe_resume, linkedin_profile (Zubair shares both SWE resume and LinkedIn)

INCOMING EMAIL GUIDELINES:
- Recruiter requesting resume and professional profile FROM Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair"
- Software engineering or tech position
- Professional recruiter tone
- 400-500 characters
- Include sender name, company, and email (company domain)

REPLY GUIDELINES:
- Zubair (the student) responds professionally
- Shares SWE resume: https://drive.google.com/file/d/1NsbClvTx1MgSHhjvdcdpdTppI5MXxtYQ/view?usp=sharing
- Shares LinkedIn: https://www.linkedin.com/in/zubair-atha/
- Expresses interest
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_10_results = process_combination(
    COMBINATION_10_PROMPT,
    combination_num=10,
    topic="Recruiter/Job Search",
    intents=['send_materials'],
    artifacts=['swe_resume', 'linkedin_profile']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 10 - ALL FORMATTED EMAILS ({len(combination_10_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_10_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 10...
✅ Generated 5 emails

📋 Validating Combination 10...

Email 1:
  ⚠️  Prospect email length: 205 (should be 400-500)
  ⚠️  Reply length: 343 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 2:
  ⚠️  Prospect email length: 178 (should be 400-500)
  ⚠️  Reply length: 326 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 3:
  ⚠️  Prospect email length: 151 (should be 400-500)
  ⚠️  Reply length: 357 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 4:
  ⚠️  Prospect email length: 152 (should be 400-500)
  ⚠️  Reply length: 360 (should be 400-500)
  ⚠️  SWE resume link not found in reply

Email 5:
  ⚠️  Prospect email length: 171 (should be 400-500)
  ⚠️  Reply length: 355 (should be 400-500)
  ⚠️  SWE resume link not found in reply

✅ Valid emails: 5/5

💾 Total emails collected: 50

📧 COMBINATION 10 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "c1caed55-e6e5-4830-9f0c-9a0818493

In [16]:
# Combination 11: Recruiter/Job Search | send_materials | portfolio, github

COMBINATION_11_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Recruiter/Job Search
INTENT: send_materials (recruiter requests portfolio/work samples from student Zubair)
ARTIFACTS: portfolio, github (Zubair shares both portfolio and GitHub)

INCOMING EMAIL GUIDELINES:
- Recruiter interested in seeing portfolio and code samples FROM Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair"
- Software engineering or developer position
- Professional recruiter tone
- 400-500 characters
- Include sender name, company, and email (company domain)

REPLY GUIDELINES:
- Zubair (the student) responds professionally
- Shares Portfolio: https://zubairatha.vercel.app/
- Shares GitHub: https://github.com/zubairatha
- Mentions relevant projects
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_11_results = process_combination(
    COMBINATION_11_PROMPT,
    combination_num=11,
    topic="Recruiter/Job Search",
    intents=['send_materials'],
    artifacts=['portfolio', 'github']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 11 - ALL FORMATTED EMAILS ({len(combination_11_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_11_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 11...
✅ Generated 5 emails

📋 Validating Combination 11...

Email 1:
  ⚠️  Prospect email length: 268 (should be 400-500)
  ⚠️  Reply length: 320 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 249 (should be 400-500)
  ⚠️  Reply length: 333 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 209 (should be 400-500)
  ⚠️  Reply length: 272 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 209 (should be 400-500)
  ⚠️  Reply length: 288 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 199 (should be 400-500)
  ⚠️  Reply length: 302 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 55

📧 COMBINATION 11 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "c7ec5e74-f581-4857-a45f-53489066d949",
  "labels": {
    "topic": "Recruiter/Job Search",
    "intents": [
      "send_materials"
    ],
    "artifacts": [
      "portfolio",
      "github"
    ]
  },
  "subject": "Opportunity at TechCorp

In [17]:
# Combination 12: Group/Event Coordination | schedule | calendly

COMBINATION_12_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Group/Event Coordination
INTENT: schedule (group/team wants to schedule meeting/event with student Zubair)
ARTIFACT: calendly (Zubair shares Calendly link)

INCOMING EMAIL GUIDELINES:
- Team member or event coordinator reaching out TO Zubair (the student) to schedule group meeting
- Address Zubair as "Zubair" or "Dear Zubair"
- Could be project meeting, team sync, event planning, etc.
- Friendly professional tone
- 400-500 characters
- Include sender name and email (university/company domain)

REPLY GUIDELINES:
- Zubair (the student) responds positively
- Shares Calendly link: https://calendly.com/za2366-columbia/30min
- Mentions availability
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_12_results = process_combination(
    COMBINATION_12_PROMPT,
    combination_num=12,
    topic="Group/Event Coordination",
    intents=['schedule'],
    artifacts=['calendly']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 12 - ALL FORMATTED EMAILS ({len(combination_12_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_12_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 12...
✅ Generated 5 emails

📋 Validating Combination 12...

Email 1:
  ⚠️  Prospect email length: 184 (should be 400-500)
  ⚠️  Reply length: 190 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 182 (should be 400-500)
  ⚠️  Reply length: 232 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 183 (should be 400-500)
  ⚠️  Reply length: 267 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 176 (should be 400-500)
  ⚠️  Reply length: 217 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 184 (should be 400-500)
  ⚠️  Reply length: 259 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 60

📧 COMBINATION 12 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "17b3d2e7-29ac-4f5b-8a50-a2140fba9c45",
  "labels": {
    "topic": "Group/Event Coordination",
    "intents": [
      "schedule"
    ],
    "artifacts": [
      "calendly"
    ]
  },
  "subject": "Project Sync-Up - Next Week?",
  "sender_e

In [18]:
# Combination 13: Group/Event Coordination | request_feedback | None

COMBINATION_13_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Group/Event Coordination
INTENT: request_feedback (team member requests feedback from student Zubair)
ARTIFACTS: None (Zubair provides feedback but doesn't send documents)

INCOMING EMAIL GUIDELINES:
- Team member asking Zubair (the student) for feedback on project, presentation, or work
- Address Zubair as "Zubair" or "Dear Zubair"
- Could be slides, proposal, event plan, etc.
- Friendly collaborative tone
- 400-500 characters
- Include sender name and email (university/company domain)

REPLY GUIDELINES:
- Zubair (the student) provides constructive feedback
- Offers to discuss further if needed
- No documents shared, just feedback in email
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_13_results = process_combination(
    COMBINATION_13_PROMPT,
    combination_num=13,
    topic="Group/Event Coordination",
    intents=['request_feedback'],
    artifacts=[]
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 13 - ALL FORMATTED EMAILS ({len(combination_13_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_13_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 13...
✅ Generated 5 emails

📋 Validating Combination 13...

Email 1:
  ⚠️  Prospect email length: 152 (should be 400-500)
  ⚠️  Reply length: 237 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 146 (should be 400-500)
  ⚠️  Reply length: 247 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 152 (should be 400-500)
  ⚠️  Reply length: 269 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 143 (should be 400-500)
  ⚠️  Reply length: 246 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 138 (should be 400-500)
  ⚠️  Reply length: 248 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 65

📧 COMBINATION 13 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "7726ae37-0b34-45a2-a19a-e9e0f94c87d6",
  "labels": {
    "topic": "Group/Event Coordination",
    "intents": [
      "request_feedback"
    ],
    "artifacts": []
  },
  "subject": "Feedback Request for Presentation",
  "sender_email": "a

In [19]:
# Combination 14: Group/Event Coordination | reschedule | zoom_link

COMBINATION_14_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Group/Event Coordination
INTENT: reschedule (team wants to reschedule meeting with student Zubair)
ARTIFACT: zoom_link (Zubair provides Zoom link for rescheduled meeting)

INCOMING EMAIL GUIDELINES:
- Team member requesting to reschedule group meeting with Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair"
- Mentions reason or new proposed time
- Friendly professional tone
- 400-500 characters
- Include sender name and email (university/company domain)

REPLY GUIDELINES:
- Zubair (the student) confirms reschedule
- Shares Zoom link: https://us05web.zoom.us/j/5756631049?pwd=2whWm7gb5MHL5GIspFbviES0GQPuyE.1
- Mentions Meeting ID: 575 663 1049 and Passcode: 4Jp5UP
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_14_results = process_combination(
    COMBINATION_14_PROMPT,
    combination_num=14,
    topic="Group/Event Coordination",
    intents=['reschedule'],
    artifacts=['zoom_link']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 14 - ALL FORMATTED EMAILS ({len(combination_14_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_14_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 14...
✅ Generated 5 emails

📋 Validating Combination 14...

Email 1:
  ⚠️  Prospect email length: 198 (should be 400-500)
  ⚠️  Reply length: 221 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 196 (should be 400-500)
  ⚠️  Reply length: 233 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 170 (should be 400-500)
  ⚠️  Reply length: 228 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 148 (should be 400-500)
  ⚠️  Reply length: 222 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 155 (should be 400-500)
  ⚠️  Reply length: 235 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 70

📧 COMBINATION 14 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "9c37ae25-57dd-441a-bdac-7655e033e04c",
  "labels": {
    "topic": "Group/Event Coordination",
    "intents": [
      "reschedule"
    ],
    "artifacts": [
      "zoom_link"
    ]
  },
  "subject": "Rescheduling our group meeting for Tues

In [20]:
# Combination 15: Group/Event Coordination | confirm, request_info | zoom_link

COMBINATION_15_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Group/Event Coordination
INTENTS: confirm, request_info (team confirms meeting AND requests information from student Zubair)
ARTIFACT: zoom_link (Zubair confirms and provides Zoom link)

INCOMING EMAIL GUIDELINES:
- Team member confirming meeting details AND asking Zubair (the student) for additional info
- Address Zubair as "Zubair" or "Dear Zubair"
- Could be confirming time and asking for agenda, materials, etc.
- Friendly professional tone
- 400-500 characters
- Include sender name and email (university/company domain)

REPLY GUIDELINES:
- Zubair (the student) confirms meeting
- Provides requested information
- Shares Zoom link: https://us05web.zoom.us/j/5756631049?pwd=2whWm7gb5MHL5GIspFbviES0GQPuyE.1
- Mentions Meeting ID: 575 663 1049 and Passcode: 4Jp5UP
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_15_results = process_combination(
    COMBINATION_15_PROMPT,
    combination_num=15,
    topic="Group/Event Coordination",
    intents=['confirm', 'request_info'],
    artifacts=['zoom_link']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 15 - ALL FORMATTED EMAILS ({len(combination_15_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_15_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 15...
✅ Generated 5 emails

📋 Validating Combination 15...

Email 1:
  ⚠️  Prospect email length: 220 (should be 400-500)
  ⚠️  Reply length: 280 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 244 (should be 400-500)
  ⚠️  Reply length: 275 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 257 (should be 400-500)
  ⚠️  Reply length: 287 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 255 (should be 400-500)
  ⚠️  Reply length: 245 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 242 (should be 400-500)
  ⚠️  Reply length: 266 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 75

📧 COMBINATION 15 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "c20f27bc-9744-4ab5-ac4f-e6ed5cd3947d",
  "labels": {
    "topic": "Group/Event Coordination",
    "intents": [
      "confirm",
      "request_info"
    ],
    "artifacts": [
      "zoom_link"
    ]
  },
  "subject": "Confirming Wednesday

In [25]:
# Combination 16: Feedback & Reviews | request_feedback | report

COMBINATION_16_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Feedback & Reviews
INTENT: request_feedback (someone requests feedback from student Zubair on a report)
ARTIFACT: report (Zubair sends a report)

INCOMING EMAIL GUIDELINES:
- Someone asking Zubair (the student) for feedback on a report or document
- Address Zubair as "Zubair" or "Dear Zubair"
- Could be colleague, professor, or team member
- Professional tone
- 400-500 characters
- Include sender name and email

REPLY GUIDELINES:
- Zubair (the student) acknowledges request
- Mentions sending report document
- Offers to discuss further
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_16_results = process_combination(
    COMBINATION_16_PROMPT,
    combination_num=16,
    topic="Feedback & Reviews",
    intents=['request_feedback'],
    artifacts=['report']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 16 - ALL FORMATTED EMAILS ({len(combination_16_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_16_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 16...
✅ Generated 5 emails

📋 Validating Combination 16...

Email 1:
  ⚠️  Prospect email length: 199 (should be 400-500)
  ⚠️  Reply length: 329 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 205 (should be 400-500)
  ⚠️  Reply length: 273 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 204 (should be 400-500)
  ⚠️  Reply length: 318 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 199 (should be 400-500)
  ⚠️  Reply length: 330 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 202 (should be 400-500)
  ⚠️  Reply length: 351 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 85

📧 COMBINATION 16 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "a17a8d05-f570-4554-80a5-dee96e6c0eab",
  "labels": {
    "topic": "Feedback & Reviews",
    "intents": [
      "request_feedback"
    ],
    "artifacts": [
      "report"
    ]
  },
  "subject": "Feedback Request on Research Report",
  "s

In [26]:
# Combination 17: Feedback & Reviews | share_feedback | draft

COMBINATION_17_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Feedback & Reviews
INTENT: share_feedback (someone shares feedback on student Zubair's work, Zubair responds with draft)
ARTIFACT: draft (Zubair sends a draft document)

INCOMING EMAIL GUIDELINES:
- Someone providing feedback TO Zubair (the student) on his work
- Address Zubair as "Zubair" or "Dear Zubair"
- Could be professor, colleague, or reviewer
- Constructive feedback tone
- 400-500 characters
- Include sender name and email

REPLY GUIDELINES:
- Zubair (the student) thanks for feedback respectfully
- Mentions sending revised draft incorporating feedback
- Appreciates the input
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_17_results = process_combination(
    COMBINATION_17_PROMPT,
    combination_num=17,
    topic="Feedback & Reviews",
    intents=['share_feedback'],
    artifacts=['draft']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 17 - ALL FORMATTED EMAILS ({len(combination_17_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_17_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 17...
✅ Generated 5 emails

📋 Validating Combination 17...

Email 1:
  ⚠️  Prospect email length: 292 (should be 400-500)
  ⚠️  Reply length: 306 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 238 (should be 400-500)
  ⚠️  Reply length: 378 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 257 (should be 400-500)
  ⚠️  Reply length: 344 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 227 (should be 400-500)
  ⚠️  Reply length: 354 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 230 (should be 400-500)
  ⚠️  Reply length: 396 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 90

📧 COMBINATION 17 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "0d82157f-5bce-4452-9185-2cd038d931fc",
  "labels": {
    "topic": "Feedback & Reviews",
    "intents": [
      "share_feedback"
    ],
    "artifacts": [
      "draft"
    ]
  },
  "subject": "Feedback on Draft Submission",
  "sender_emai

In [23]:
# Combination 18: Feedback & Reviews | request_info | github

COMBINATION_18_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Feedback & Reviews
INTENT: request_info (someone requests information from student Zubair about a project)
ARTIFACT: github (Zubair shares GitHub repository)

INCOMING EMAIL GUIDELINES:
- Someone asking Zubair (the student) about a project or code repository
- Address Zubair as "Zubair" or "Dear Zubair"
- Could be colleague, professor, or collaborator
- Professional tone
- 400-500 characters
- Include sender name and email

REPLY GUIDELINES:
- Zubair (the student) responds helpfully
- Shares GitHub link: https://github.com/zubairatha
- Mentions relevant project or repository
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_18_results = process_combination(
    COMBINATION_18_PROMPT,
    combination_num=18,
    topic="Feedback & Reviews",
    intents=['request_info'],
    artifacts=['github']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 18 - ALL FORMATTED EMAILS ({len(combination_18_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_18_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 18...
✅ Generated 5 emails

📋 Validating Combination 18...

Email 1:
  ⚠️  Prospect email length: 343 (should be 400-500)
  ⚠️  Reply length: 399 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 303 (should be 400-500)
  ⚠️  Reply length: 371 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 290 (should be 400-500)
  ⚠️  Reply length: 365 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 312 (should be 400-500)
  ⚠️  Reply length: 331 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 284 (should be 400-500)
  ⚠️  Reply length: 350 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 80

📧 COMBINATION 18 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "df65de2b-1c6b-4fd2-a566-1418b59aed65",
  "labels": {
    "topic": "Feedback & Reviews",
    "intents": [
      "request_info"
    ],
    "artifacts": [
      "github"
    ]
  },
  "subject": "Review of Code Repository Needed",
  "sender_e

In [28]:
len(all_emails_formatted)

90

In [29]:
# Combination 19: Scheduling | schedule | calendly

COMBINATION_19_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Scheduling
INTENT: schedule (someone wants to schedule a meeting with student Zubair)
ARTIFACT: calendly (Zubair shares Calendly link)

INCOMING EMAIL GUIDELINES:
- Generic scheduling request TO Zubair (the student) (not specific to professor/recruiter/group)
- Address Zubair as "Zubair" or "Dear Zubair"
- Could be networking, general meeting, consultation, etc.
- Professional tone
- 400-500 characters
- Include sender name and email

REPLY GUIDELINES:
- Zubair (the student) responds positively
- Shares Calendly link: https://calendly.com/za2366-columbia/30min
- Mentions availability
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_19_results = process_combination(
    COMBINATION_19_PROMPT,
    combination_num=19,
    topic="Scheduling",
    intents=['schedule'],
    artifacts=['calendly']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 19 - ALL FORMATTED EMAILS ({len(combination_19_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_19_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 19...
✅ Generated 5 emails

📋 Validating Combination 19...

Email 1:
  ⚠️  Prospect email length: 250 (should be 400-500)
  ⚠️  Reply length: 325 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 186 (should be 400-500)
  ⚠️  Reply length: 322 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 191 (should be 400-500)
  ⚠️  Reply length: 320 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 156 (should be 400-500)
  ⚠️  Reply length: 334 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 154 (should be 400-500)
  ⚠️  Reply length: 310 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 95

📧 COMBINATION 19 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "711e8a19-6c94-40ef-9545-dd8450ddfb15",
  "labels": {
    "topic": "Scheduling",
    "intents": [
      "schedule"
    ],
    "artifacts": [
      "calendly"
    ]
  },
  "subject": "Meeting Request - Collaboration Discussion",
  "sender_e

In [30]:
# Combination 20: Scheduling | reschedule | zoom_link, phone_number

COMBINATION_20_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Scheduling
INTENT: reschedule (someone wants to reschedule a meeting with student Zubair)
ARTIFACTS: zoom_link, phone_number (Zubair provides both Zoom link and phone number)

INCOMING EMAIL GUIDELINES:
- Generic reschedule request TO Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair"
- Mentions conflict or need to change time
- Professional tone
- 400-500 characters
- Include sender name and email

REPLY GUIDELINES:
- Zubair (the student) confirms reschedule
- Shares Zoom link: https://us05web.zoom.us/j/5756631049?pwd=2whWm7gb5MHL5GIspFbviES0GQPuyE.1
- Mentions Meeting ID: 575 663 1049 and Passcode: 4Jp5UP
- Provides phone number: +16463925601 (as backup option)
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_20_results = process_combination(
    COMBINATION_20_PROMPT,
    combination_num=20,
    topic="Scheduling",
    intents=['reschedule'],
    artifacts=['zoom_link', 'phone_number']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 20 - ALL FORMATTED EMAILS ({len(combination_20_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_20_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 20...
✅ Generated 5 emails

📋 Validating Combination 20...

Email 1:
  ⚠️  Prospect email length: 192 (should be 400-500)
  ⚠️  Reply length: 291 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 197 (should be 400-500)
  ⚠️  Reply length: 301 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 182 (should be 400-500)
  ⚠️  Reply length: 287 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 185 (should be 400-500)
  ⚠️  Reply length: 289 (should be 400-500)

Email 5:
  ⚠️  Prospect email length: 202 (should be 400-500)
  ⚠️  Reply length: 316 (should be 400-500)

✅ Valid emails: 5/5

💾 Total emails collected: 100

📧 COMBINATION 20 - ALL FORMATTED EMAILS (5 emails)

--- Email 1 ---
{
  "id": "9baa9bac-121b-4d16-821b-9ae5a87ae03c",
  "labels": {
    "topic": "Scheduling",
    "intents": [
      "reschedule"
    ],
    "artifacts": [
      "zoom_link",
      "phone_number"
    ]
  },
  "subject": "Rescheduling Our Meeting",
  "

In [31]:
# Combination 21: Scheduling | confirm | phone_number

COMBINATION_21_PROMPT = """
Generate 5 different email pairs (incoming email + reply) with these specifications:

IMPORTANT CONTEXT: Zubair is a GRADUATE STUDENT. The incoming email is addressed TO Zubair (the student), and the reply is FROM Zubair (the student).

TOPIC: Scheduling
INTENT: confirm (someone confirms a meeting with student Zubair and requests contact)
ARTIFACT: phone_number (Zubair shares phone number)

INCOMING EMAIL GUIDELINES:
- Someone confirming a scheduled meeting with Zubair (the student)
- Address Zubair as "Zubair" or "Dear Zubair"
- May request phone number for call or backup contact
- Professional tone
- 400-500 characters
- Include sender name and email

REPLY GUIDELINES:
- Zubair (the student) confirms meeting
- Shares phone number: +16463925601
- Mentions availability for the call
- 400-500 characters
- Must end with: "Best Regards,\nZubair"

OUTPUT FORMAT (JSON):
{
  "email_1": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_2": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_3": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_4": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."},
  "email_5": {"subject": "...", "sender_email": "...", "prospect_email": "...", "reply": "..."}
}
"""

combination_21_results = process_combination(
    COMBINATION_21_PROMPT,
    combination_num=21,
    topic="Scheduling",
    intents=['confirm'],
    artifacts=['phone_number']
)

# Print all formatted emails for this combination
print(f"\n{'='*70}")
print(f"📧 COMBINATION 21 - ALL FORMATTED EMAILS ({len(combination_21_results)} emails)")
print(f"{'='*70}")
for i, email in enumerate(combination_21_results, 1):
    print(f"\n--- Email {i} ---")
    print(json.dumps(email, indent=2))
print(f"\n{'='*70}\n")



🔄 Generating emails for Combination 21...
✅ Generated 5 emails

📋 Validating Combination 21...

Email 1:
  ⚠️  Prospect email length: 252 (should be 400-500)
  ⚠️  Reply length: 222 (should be 400-500)

Email 2:
  ⚠️  Prospect email length: 254 (should be 400-500)
  ⚠️  Reply length: 221 (should be 400-500)

Email 3:
  ⚠️  Prospect email length: 285 (should be 400-500)
  ⚠️  Reply length: 225 (should be 400-500)

Email 4:
  ⚠️  Prospect email length: 227 (should be 400-500)
  ⚠️  Reply length: 236 (should be 400-500)
❌ Missing email_5

✅ Valid emails: 4/5

💾 Total emails collected: 104

📧 COMBINATION 21 - ALL FORMATTED EMAILS (4 emails)

--- Email 1 ---
{
  "id": "fca35d4a-2ee6-41fa-bf29-34a35a2e248d",
  "labels": {
    "topic": "Scheduling",
    "intents": [
      "confirm"
    ],
    "artifacts": [
      "phone_number"
    ]
  },
  "subject": "Reminder for Tomorrow's Meeting",
  "sender_email": "john.doe@example.com",
  "prospect_email": "Hi Zubair,\n\nThis is a friendly reminder th

In [32]:
# Save all generated emails to JSONL file
print(f"\n💾 Saving {len(all_emails_formatted)} emails to file...")

output_file = "generated_email_pairs.jsonl"
with open(output_file, 'w', encoding='utf-8') as f:
    for email in all_emails_formatted:
        f.write(json.dumps(email, ensure_ascii=False) + '\n')

print(f"✅ Saved to {output_file}")
print(f"📊 Summary:")
print(f"   - Total emails: {len(all_emails_formatted)}")
print(f"   - Expected: 105 (21 combinations × 5 emails)")
print(f"   - Coverage: {len(all_emails_formatted)/105*100:.1f}%")

# Also save as JSON for easy inspection
with open("generated_email_pairs.json", 'w', encoding='utf-8') as f:
    json.dump(all_emails_formatted, f, indent=2, ensure_ascii=False)

print(f"✅ Also saved to generated_email_pairs.json")



💾 Saving 104 emails to file...
✅ Saved to generated_email_pairs.jsonl
📊 Summary:
   - Total emails: 104
   - Expected: 105 (21 combinations × 5 emails)
   - Coverage: 99.0%
✅ Also saved to generated_email_pairs.json
